# 拡散モデルを体験する

## 1. これからやること

ここでは、[DDPM](https://arxiv.org/pdf/2006.11239)という拡散モデルを用いた画像生成の過程を体験します。
デモの実装には[pierrot-lc](https://github.com/pierrot-lc)氏の[anime-diffusion](https://github.com/pierrot-lc/anime-diffusion/tree/main)を使用しました。

### 拡散モデルのお気持ち

そもそも拡散モデルとはなんなのでしょうか？
拡散モデルとは、
- データにノイズを**付加**していく過程 (順拡散過程)
- ノイズを**除去**していく過程 (逆拡散過程)
の2つの過程により、データの生成を実現するモデルのことです。

![DDPM diffusion process](https://sushant-kumar.com/blog/ddpm-diffusion-process.png)

(引用: sushant-kumar, https://sushant-kumar.com/blog/ddpm-denoising-diffusion-probabilistic-models)

図のように、順拡散過程では画像データ$x_0$にノイズを時刻が進むにつれて徐々に付加していくことで、最終的に完全なノイズ$x_T$を得ることができます。
一方、逆拡散過程では、ノイズが付加された画像データ$x_t$からノイズを除去していくことで、最終的に元の画像データ$x_0$を復元することを目指します。

拡散モデルの**学習**とは、この逆拡散過程を学習することを指します。
つまり、ノイズだらけの画像データ$x_t$からノイズを除去していく過程を学習することで、元の画像データ$x_0$を復元する能力を獲得させることが目的となります。

この能力を獲得したモデルに、ランダムにサンプリングしたノイズデータを入力することで、逆拡散過程を通じて画像データを生成することができるようになります。

## 2. 逆拡散過程を可視化してみる

逆拡散過程により、データがどのように生成されていくのかを可視化してみましょう。
以下のコードブロックを順に実行してみてください。

### セットアップ

In [1]:
%%capture
!git lfs install
!git clone https://github.com/pierrot-lc/anime-diffusion.git /content/anime-diffusion
!pip install equinox einops jaxtyping hydra-core beartype optax

In [2]:
%cd anime-diffusion
%ls

/content/anime-diffusion
configs/    flake.lock  generate.py  LICENCE  pdm.lock        README.md
final-run/  flake.nix   justfile     main.py  pyproject.toml  src/


In [3]:
from PIL.features import version
import equinox as eqx
import jax
import jax.numpy as jnp
import jax.random as random
import numpy as np
from pathlib import Path
from PIL import Image
from einops import rearrange
from IPython.display import display, Image as IPImage
from configs import (
    AnimeDatasetConfig,
    DiffusionConfig,
    MainConfig,
    ModelConfig,
    TrainerConfig,
)
from src.model import UViT
from src.diffusion import decode_all, scheduler, CLIP
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra


In [4]:

GlobalHydra.instance().clear()
exp_dir = Path("/content/anime-diffusion/final-run").absolute()
with initialize_config_dir(str(exp_dir / ".hydra"), version_base="1.1"):
    cfg = compose("config.yaml")
print(cfg)

{'mode': 'online', 'dataset': {'image_size': 96, 'seed': 0}, 'diffusion': {'steps': 1000}, 'model': {'patch_size': 8, 'd_model': 256, 'num_heads': 8, 'num_layers': 16, 'seed': 100}, 'trainer': {'batch_size': 64, 'total_iters': 1000000.0, 'evaluate_steps': 3000, 'learning_rate': 0.0003, 'preload_data': True, 'seed': 42}}


モデルのロード

In [5]:
config = MainConfig(
    dataset=AnimeDatasetConfig(**cfg.dataset),
    diffusion=DiffusionConfig(**cfg.diffusion),
    model=ModelConfig(**cfg.model),
    trainer=TrainerConfig(**cfg.trainer),
    mode=cfg.mode,
)

schedule = scheduler(config.diffusion.steps)

model = UViT(
    num_channels=config.dataset.n_channels,
    num_positions=(config.dataset.image_size // config.model.patch_size) ** 2,
    num_timesteps=len(schedule),
    patch_size=config.model.patch_size,
    d_model=config.model.d_model,
    num_heads=config.model.num_heads,
    num_layers=config.model.num_layers,
    key=random.key(config.model.seed),
)
model = eqx.tree_deserialise_leaves(exp_dir / "checkpoint.eqx", model)

print(f"Model loaded  —  device: {jax.devices()[0]}")

Model loaded  —  device: cuda:0


生成

In [6]:
from generate import to_gif, make_grid

N_IMAGES = 9
SEED = 42

image_shape = (
    config.dataset.n_channels,
    config.dataset.image_size,
    config.dataset.image_size,
)

model_inf = eqx.nn.inference_mode(model)
decode_fn = jax.vmap(decode_all, in_axes=(0, None, None, 0))
make_grid_fn = jax.vmap(make_grid)

sk_decode, sk_normal = random.split(random.key(SEED))
xT = random.normal(sk_normal, (N_IMAGES, *image_shape))
xT = jnp.clip(xT, -CLIP, CLIP)

print("Running backward diffusion … (this may take a minute)")
x_dec = decode_fn(xT, model_inf, schedule, random.split(sk_decode, N_IMAGES))
x_dec = rearrange(x_dec, "b s c h w -> s b c h w")
grid = make_grid_fn(x_dec)

# Save outputs
out_gif  = Path("/content/generation.gif")
out_png  = Path("/content/generation.png")
Image.fromarray(np.array(grid[-1])).save(out_png)
to_gif(rearrange(grid, "s h w c -> s c h w"), out_gif)

Running backward diffusion … (this may take a minute)


逆拡散過程の様子

In [7]:
display(IPImage(filename="/content/generation.gif"))

Output hidden; open in https://colab.research.google.com to view.

## 3. おわりに

今回は、拡散モデルの基本的な考え方と、逆拡散過程の様子を確認してみました。

ところで、逆拡散過程では、ノイズが付加された画像データからノイズを除去していくことで、元の画像データを復元することを目指すと説明しました。
しかし、なぜノイズを除去していくことで、元の画像データを復元することができるのでしょうか？ 直感的には妥当なアイデアのように思えますが、
数学的に考えると、ノイズを予測することで元の画像データの確率分布を学習できるというのは非自明なことのように思えます。

このあたりの理論的な背景については、[ネットの解説記事](https://zenn.dev/doctorin/articles/diffusion_models#%E3%81%AF%E3%81%98%E3%82%81%E3%81%AB)
や、書籍[1](https://www.amazon.co.jp/%E6%8B%A1%E6%95%A3%E3%83%A2%E3%83%87%E3%83%AB-%E3%83%87%E3%83%BC%E3%82%BF%E7%94%9F%E6%88%90%E6%8A%80%E8%A1%93%E3%81%AE%E6%95%B0%E7%90%86-%E5%B2%A1%E9%87%8E%E5%8E%9F-%E5%A4%A7%E8%BC%94/dp/400006343X?&linkCode=sl1&tag=doctorin-22&linkId=990451e5c4da9edadbeb974a51d37400&language=ja_JP&ref_=as_li_ss_tl),
[2](https://www.oreilly.co.jp/books/9784814400591/)
などで詳しく解説されているので、興味のある方はぜひ読んでみてください。

もし今回のデモが「面白い！」と感じた方は、ぜひkaggle班で共に機械学習の世界を探求してみましょう！